In [3]:
import geopandas as gpd
import folium
from folium.features import GeoJsonPopup
from pathlib import Path

gpkg_path = Path("VAL_ОКН.gpkg")      # при необходимости укажи полный путь
out_html  = Path("VAL_OKN_map.html")

gdf = gpd.read_file(gpkg_path)

# CRS -> WGS84
if gdf.crs is None:
    raise ValueError("CRS отсутствует. Сначала задай его: gdf = gdf.set_crs('EPSG:XXXX')")
gdf = gdf.to_crs(epsg=4326)

# Центр карты: по центроидам (работает для любых геометрий)
centroids = gdf.geometry.centroid
center_lat = float(centroids.y.mean())
center_lon = float(centroids.x.mean())

m = folium.Map(location=[center_lat, center_lon], zoom_start=12, control_scale=True)

# Поля для попапа (вся атрибутивка кроме geometry)
fields = [c for c in gdf.columns if c != "geometry"]

folium.GeoJson(
    gdf,
    name="VAL_ОКН",
    popup=GeoJsonPopup(fields=fields, aliases=fields, labels=True, localize=True),
).add_to(m)

folium.LayerControl().add_to(m)
m.save(out_html.as_posix())

print(f"Saved: {out_html.resolve()}")

Saved: C:\Users\Huawei\VU\VAL_OKN_map.html


C:\Users\Huawei\AppData\Local\Temp\ipykernel_19492\2681586919.py:17: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  centroids = gdf.geometry.centroid


In [5]:
import geopandas as gpd
import folium
from folium.features import GeoJsonPopup, GeoJsonTooltip
from branca.element import MacroElement
from jinja2 import Template
from pathlib import Path

gpkg_path = Path("VAL_ОКН.gpkg")      # при необходимости укажи полный путь
out_html  = Path("VAL_OKN_map.html")

gdf = gpd.read_file(gpkg_path)

# CRS -> WGS84
if gdf.crs is None:
    raise ValueError("CRS отсутствует. Сначала задай его: gdf = gdf.set_crs('EPSG:XXXX')")
gdf = gdf.to_crs(epsg=4326)

# Если геометрия не Point/MultiPoint — отображаем точку внутри объекта (для клика)
if not gdf.geometry.geom_type.isin(["Point", "MultiPoint"]).all():
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.representative_point()

# Центр карты
center = gdf.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=12, control_scale=True)

# Поля попапа: вся атрибутика кроме geometry
fields = [c for c in gdf.columns if c != "geometry"]

# Цвета по reg_stat
color_map = {
    1: "#d73027",  # федеральные
    2: "#4575b4",  # региональные
    8: "#1a9850",  # местные (выявленные)
}

def _norm_reg_stat(v):
    try:
        return int(float(v))
    except Exception:
        return None

def style_fn(feature):
    rs = _norm_reg_stat(feature["properties"].get("reg_stat"))
    col = color_map.get(rs, "#666666")
    return {
        "color": col,
        "fillColor": col,
        "fillOpacity": 0.85,
        "weight": 1,
        "radius": 5,
    }

# Слой: точки + попап с атрибутивной таблицей (по клику)
folium.GeoJson(
    gdf,
    name="VAL_ОКН",
    marker=folium.CircleMarker(),  # включает отрисовку точек
    style_function=style_fn,
    highlight_function=lambda f: {"weight": 3},
    tooltip=GeoJsonTooltip(
        fields=(["reg_stat"] if "reg_stat" in fields else fields[:1]),
        aliases=(["reg_stat"] if "reg_stat" in fields else fields[:1]),
        sticky=True,
    ),
    popup=GeoJsonPopup(
        fields=fields,
        aliases=fields,
        labels=True,
        localize=True,
        max_width=700,
    ),
).add_to(m)

# Легенда
legend_html = """
{% macro html(this, kwargs) %}
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    z-index: 9999;
    background: white;
    padding: 10px 12px;
    border: 1px solid #999;
    border-radius: 6px;
    font-size: 13px;
">
  <div style="font-weight: 600; margin-bottom: 6px;">reg_stat</div>
  <div><span style="display:inline-block;width:12px;height:12px;background:#d73027;margin-right:8px;border:1px solid #555;"></span>1 — федеральные</div>
  <div><span style="display:inline-block;width:12px;height:12px;background:#4575b4;margin-right:8px;border:1px solid #555;"></span>2 — региональные</div>
  <div><span style="display:inline-block;width:12px;height:12px;background:#1a9850;margin-right:8px;border:1px solid #555;"></span>8 — местные (выявленные)</div>
</div>
{% endmacro %}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl().add_to(m)
m.save(out_html.as_posix())

print(f"Saved: {out_html.resolve()}")

Saved: C:\Users\Huawei\VU\VAL_OKN_map.html


C:\Users\Huawei\AppData\Local\Temp\ipykernel_19492\1004546177.py:24: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = gdf.geometry.unary_union.centroid


In [7]:
import geopandas as gpd
import folium
from folium.features import GeoJsonPopup, GeoJsonTooltip
from branca.element import MacroElement
from jinja2 import Template
from pathlib import Path
import json

gpkg_path = Path("VAL_ОКН.gpkg")      # при необходимости укажи полный путь
out_html  = Path("VAL_OKN_map.html")

gdf = gpd.read_file(gpkg_path)

# CRS -> WGS84
if gdf.crs is None:
    raise ValueError("CRS отсутствует. Сначала задай его: gdf = gdf.set_crs('EPSG:XXXX')")
gdf = gdf.to_crs(epsg=4326)

# Кликабельные точки для любых геометрий
if not gdf.geometry.geom_type.isin(["Point", "MultiPoint"]).all():
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.representative_point()

# Центр карты
center = gdf.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=12, control_scale=True)

# Поля попапа: вся атрибутика кроме geometry
fields = [c for c in gdf.columns if c != "geometry"]

# Цвета по reg_stat
color_map = {1: "#d73027", 2: "#4575b4", 8: "#1a9850"}

def norm_reg_stat(v):
    try:
        return int(float(v))
    except Exception:
        return None

def get_color(rs):
    return color_map.get(rs, "#666666")

# Сгруппировать по reg_stat и добавить как CircleMarker (тогда попап точно работает)
for rs_value, sub in gdf.groupby(gdf["reg_stat"].apply(norm_reg_stat), dropna=False):
    col = get_color(rs_value)

    fg_name = (
        "1 — федеральные" if rs_value == 1 else
        "2 — региональные" if rs_value == 2 else
        "8 — местные (выявленные)" if rs_value == 8 else
        "прочее/нет значения"
    )
    fg = folium.FeatureGroup(name=fg_name)

    for _, row in sub.iterrows():
        props = {k: (None if (row[k] != row[k]) else row[k]) for k in fields}  # NaN -> None

        # HTML-таблица атрибутов
        rows_html = "".join(
            f"<tr><th style='text-align:left;padding:2px 6px;border-bottom:1px solid #eee;'>{k}</th>"
            f"<td style='padding:2px 6px;border-bottom:1px solid #eee;'>{props.get(k,'')}</td></tr>"
            for k in fields
        )
        popup_html = f"<table style='border-collapse:collapse;font-size:12px;width:100%;'>{rows_html}</table>"

        geom = row.geometry
        # для MultiPoint берем первую точку
        if geom.geom_type == "MultiPoint":
            geom = list(geom.geoms)[0]

        folium.CircleMarker(
            location=[geom.y, geom.x],
            radius=6,
            color=col,
            weight=1,
            fill=True,
            fill_color=col,
            fill_opacity=0.9,
            popup=folium.Popup(popup_html, max_width=800),
            tooltip=str(props.get("reg_stat", rs_value)),
        ).add_to(fg)

    fg.add_to(m)

# Легенда (статичная)
legend_html = """
{% macro html(this, kwargs) %}
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    z-index: 9999;
    background: white;
    padding: 10px 12px;
    border: 1px solid #999;
    border-radius: 6px;
    font-size: 13px;
">
  <div style="font-weight: 600; margin-bottom: 6px;">reg_stat</div>
  <div><span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#d73027;margin-right:8px;border:1px solid #555;"></span>1 — федеральные</div>
  <div><span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#4575b4;margin-right:8px;border:1px solid #555;"></span>2 — региональные</div>
  <div><span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#1a9850;margin-right:8px;border:1px solid #555;"></span>8 — местные (выявленные)</div>
</div>
{% endmacro %}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl(collapsed=False).add_to(m)
m.save(out_html.as_posix())

print(f"Saved: {out_html.resolve()}")


C:\Users\Huawei\AppData\Local\Temp\ipykernel_19492\3865421869.py:25: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = gdf.geometry.unary_union.centroid


Saved: C:\Users\Huawei\VU\VAL_OKN_map.html


In [9]:
import geopandas as gpd
import folium
from branca.element import MacroElement
from jinja2 import Template
from pathlib import Path
import math

gpkg_path = Path("VAL_ОКН.gpkg")      # при необходимости укажи полный путь
out_html  = Path("VAL_OKN_map.html")

gdf = gpd.read_file(gpkg_path)

# CRS -> WGS84
if gdf.crs is None:
    raise ValueError("CRS отсутствует. Сначала задай CRS: gdf = gdf.set_crs('EPSG:XXXX')")
gdf = gdf.to_crs(epsg=4326)

# Кликабельные точки для любых геометрий
if not gdf.geometry.geom_type.isin(["Point", "MultiPoint"]).all():
    gdf = gdf.copy()
    gdf["geometry"] = gdf.geometry.representative_point()

# центр карты
center = gdf.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=12, control_scale=True)

# цвет по reg_stat
color_map = {1: "#d73027", 2: "#4575b4", 8: "#1a9850"}

def norm_reg_stat(v):
    try:
        if v is None or (isinstance(v, float) and math.isnan(v)):
            return None
        return int(float(v))
    except Exception:
        return None

def get_color(rs):
    return color_map.get(rs, "#666666")

# только нужные атрибуты для выноски
needed_fields = ["naim", "class_name"]

def safe_val(v):
    if v is None:
        return ""
    if isinstance(v, float) and math.isnan(v):
        return ""
    return str(v)

def callout_html(row):
    # выноска: названия полей + значения
    parts = []
    for f in needed_fields:
        if f in row.index:
            parts.append(
                f"<div style='margin:2px 0;'><b>{f}:</b> {safe_val(row[f])}</div>"
            )
    inner = "".join(parts) if parts else "<div>Нет полей naim / class_name</div>"
    return f"""
    <div style="
        min-width: 220px;
        max-width: 360px;
        font-size: 12px;
        line-height: 1.25;
    ">
      {inner}
    </div>
    """

# группировка по reg_stat (слои в легенде)
for rs_value, sub in gdf.groupby(gdf["reg_stat"].apply(norm_reg_stat), dropna=False):
    col = get_color(rs_value)
    fg_name = (
        "1 — федеральные" if rs_value == 1 else
        "2 — региональные" if rs_value == 2 else
        "8 — местные (выявленные)" if rs_value == 8 else
        "прочее/нет значения"
    )
    fg = folium.FeatureGroup(name=fg_name)

    for _, row in sub.iterrows():
        geom = row.geometry
        if geom.geom_type == "MultiPoint":
            geom = list(geom.geoms)[0]

        popup = folium.Popup(callout_html(row), max_width=420)

        folium.CircleMarker(
            location=[geom.y, geom.x],
            radius=6,
            color=col,
            weight=1,
            fill=True,
            fill_color=col,
            fill_opacity=0.9,
            popup=popup,                 # клик -> выноска с naim и class_name
            tooltip=safe_val(row.get("naim", "")),  # наведение -> naim (если есть)
        ).add_to(fg)

    fg.add_to(m)

# статичная легенда
legend_html = """
{% macro html(this, kwargs) %}
<div style="
    position: fixed;
    bottom: 30px;
    left: 30px;
    z-index: 9999;
    background: white;
    padding: 10px 12px;
    border: 1px solid #999;
    border-radius: 6px;
    font-size: 13px;
">
  <div style="font-weight: 600; margin-bottom: 6px;">reg_stat</div>
  <div><span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#d73027;margin-right:8px;border:1px solid #555;"></span>1 — федеральные</div>
  <div><span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#4575b4;margin-right:8px;border:1px solid #555;"></span>2 — региональные</div>
  <div><span style="display:inline-block;width:12px;height:12px;border-radius:50%;background:#1a9850;margin-right:8px;border:1px solid #555;"></span>8 — местные (выявленные)</div>
</div>
{% endmacro %}
"""
legend = MacroElement()
legend._template = Template(legend_html)
m.get_root().add_child(legend)

folium.LayerControl(collapsed=False).add_to(m)
m.save(out_html.as_posix())

print(f"Saved: {out_html.resolve()}")

C:\Users\Huawei\AppData\Local\Temp\ipykernel_19492\304041204.py:24: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  center = gdf.geometry.unary_union.centroid


Saved: C:\Users\Huawei\VU\VAL_OKN_map.html
